In [7]:
import os
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langgraph.types import interrupt, Command

In [8]:
@tool
def purchase_stock(ticker: str, quantity: int, price: float) -> str:
    """
    Purchase stock for a particular ticker.

    Args:
        ticker: Stock ticker symbol, e.g. AAPL or RELIANCE.NS.
        quantity: Number of shares to purchase.
        price: Price per share at which the simulated purchase is made.
    """
    if quantity <= 0:
        return "Quantity must be greater than 0."
    if price <= 0:
        return "Price must be greater than 0."

    # hit interrupt
    decision = interrupt(f"Approve buying {quantity} shares of {ticker}? Yes/No ")

    if decision == "No":
        return f"Purchase of {ticker} stocks declined by user"

    total_value = quantity * price
    return (
        f"Dummy purchase successful!\n"
        f"Ticker: {ticker.upper()}\n"
        f"Quantity: {quantity}\n"
        f"Price per share: {price:.2f}\n"
        f"Total value: {total_value:.2f}"
    )

In [9]:
class State(TypedDict):
    result: str


def purchase_node(state: State):
    result = purchase_stock.invoke({
        "ticker": "AAPL",
        "quantity": 10,
        "price": 230.50
    })
    return {"result": result}


graph_builder = StateGraph(State)
graph_builder.add_node("purchase", purchase_node)
graph_builder.add_edge(START, "purchase")
graph_builder.add_edge("purchase", END)
checkpointer = MemorySaver()

graph = graph_builder.compile(
    checkpointer=checkpointer
)

In [10]:
config = {
    "configurable": {
        "thread_id": "test-purchase-1"
    }
}

response_state = graph.invoke(
    input={},
    config=config
)

print(response_state)

{'__interrupt__': [Interrupt(value='Approve buying 10 shares of AAPL? Yes/No ', id='d7a7c8e7a2a3e779a69805bb3645df3b')]}


In [13]:
response_state["__interrupt__"][0].value

'Approve buying 10 shares of AAPL? Yes/No '